- Used a weather API to retrieve Madrid weather data for the traffic dataset’s time period and add it to the original three features.
- Added three weather features:
    - Temperature at 2 metres above ground, in °C
    - Relative humidity at 2 metres above ground, in %
    - Precipitation, in mm

In [ ]:
import shutil
import h5py
import numpy as np
import requests

from pathlib import Path

matches = list(Path("/data/mmtd").rglob("fixed_sensor_data.h5"))
print("Found:", matches)

if not matches:
    raise FileNotFoundError("fixed_sensor_data.h5 was not found under /content")

source_path = str(matches[0])
output_path = str(matches[0].with_name("fixed_sensor_weather.h5"))

print("Reading from:", source_path)
print("Saving to:", output_path)

# source_path = "data/mmtd/fixed_sensor_data.h5"
# output_path = "data/mmtd/fixed_sensor_weather.h5"

# Get hourly historical weather for Madrid, August 2024
response = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": 40.4168,
        "longitude": -3.7038,
        "start_date": "2024-08-01",
        "end_date": "2024-08-31",
        "hourly": "temperature_2m,relative_humidity_2m,precipitation",
        "timezone": "Europe/Madrid",
    },
    timeout=60,
)
response.raise_for_status()
hourly = response.json()["hourly"]

weather_hourly = np.column_stack([
    hourly["temperature_2m"],
    hourly["relative_humidity_2m"],
    hourly["precipitation"],
]).astype(np.float32)

# Repeat each hourly reading for its four 15-minute traffic intervals
weather_15min = np.repeat(weather_hourly, 4, axis=0)

with h5py.File(source_path, "r") as f:
    traffic = f["data/values"][:].astype(np.float32)
    old_features = [
        x.decode() if isinstance(x, bytes) else str(x)
        for x in f["metadata/features"][:]
    ]

T, N, F = traffic.shape
assert T == len(weather_15min), (
    f"Traffic has {T} time steps but weather has {len(weather_15min)}. "
    "Check that the file covers Aug 1–31, 2024 at 15-minute intervals."
)

# Give the same city-level weather to every sensor at each timestamp
weather_all_sensors = np.broadcast_to(
    weather_15min[:, None, :], (T, N, 3)
).copy()

combined = np.concatenate([traffic, weather_all_sensors], axis=2)
new_features = old_features + [
    "temperature_2m_C",
    "relative_humidity_2m_percent",
    "precipitation_mm",
]

# Copy the original, then replace values and feature names in the copy
shutil.copy2(source_path, output_path)

with h5py.File(output_path, "r+") as f:
    values_attrs = dict(f["data/values"].attrs)
    del f["data/values"]
    new_values = f["data"].create_dataset(
        "values", data=combined, compression="gzip", compression_opts=4
    )
    for key, value in values_attrs.items():
        new_values.attrs[key] = value

    del f["metadata/features"]
    f["metadata"].create_dataset(
        "features",
        data=np.asarray(new_features, dtype=h5py.string_dtype("utf-8")),
    )

print("Saved:", output_path)
print("New values shape:", combined.shape)
print("Features:", new_features)